**Vanguard**  
**Jaydon Hylton (2210144)** , **Chadwick Cox (1800729)** , **Semoy Smith (1505625)**

# Part 1: Workspace Setup, Git Integration, and Azure gpt-4.1-mini Deployment

 **This notebook initializes the `uv` workspace, documents Git collaboration, deploys **gpt-4.1-mini** on Azure AI Foundry, and proves dual authentication with token-budget guardrails (`temperature=0.0`, `max_tokens=256`).**


In [ ]:
%pip install -q azure-identity langchain-openai langchain-core python-dotenv

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI

TEMPERATURE = 0.0
TOKEN_BUDGET = 256
AuthMode = Literal["local", "identity"]

load_dotenv(Path.cwd() / ".env")
try:
    from google.colab import userdata

    for name in (
        "AZURE_AI_FOUNDRY_ENDPOINT",
        "AZURE_AI_FOUNDRY_KEY",
        "AZURE_MODEL_NAME",
        "AZURE_OPENAI_API_VERSION",
        "AZURE_AUTH_MODE",
    ):
        if not os.getenv(name):
            try:
                os.environ[name] = userdata.get(name)
            except Exception:
                pass
except ImportError:
    pass


@dataclass(frozen=True)
class AzureSettings:
    endpoint: str
    model_name: str
    api_version: str
    auth_mode: AuthMode
    api_key: str | None


def load_settings(auth_mode: AuthMode | None = None) -> AzureSettings:
    endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT", "").strip().rstrip("/")
    if not endpoint:
        raise ValueError("AZURE_AI_FOUNDRY_ENDPOINT is required")
    mode = (auth_mode or os.getenv("AZURE_AUTH_MODE", "local")).strip().lower()
    if mode not in ("local", "identity"):
        raise ValueError("AZURE_AUTH_MODE must be local or identity")
    api_key = os.getenv("AZURE_AI_FOUNDRY_KEY")
    if mode == "identity":
        api_key = None
    elif not api_key:
        raise ValueError("AZURE_AI_FOUNDRY_KEY is required for local authentication")
    return AzureSettings(
        endpoint=endpoint,
        model_name=os.getenv("AZURE_MODEL_NAME", "gpt-4.1-mini").strip(),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21").strip(),
        auth_mode=mode,
        api_key=api_key,
    )


def build_chat_model(auth_mode: AuthMode | None = None) -> AzureChatOpenAI:
    cfg = load_settings(auth_mode)
    kwargs = {
        "azure_endpoint": cfg.endpoint,
        "azure_deployment": cfg.model_name,
        "api_version": cfg.api_version,
        "temperature": TEMPERATURE,
        "max_tokens": TOKEN_BUDGET,
    }
    if cfg.auth_mode == "identity":
        kwargs["azure_ad_token_provider"] = get_bearer_token_provider(
            DefaultAzureCredential(),
            "https://cognitiveservices.azure.com/.default",
        )
    else:
        kwargs["api_key"] = cfg.api_key
    return AzureChatOpenAI(**kwargs)


cfg = load_settings("local")
print(cfg.endpoint, cfg.model_name, cfg.api_version, cfg.auth_mode)
print(f"temperature={TEMPERATURE} max_tokens={TOKEN_BUDGET} key_loaded={bool(cfg.api_key)}")

## Local environment protocol


In [ ]:
print(
    build_chat_model("local")
    .invoke("Reply with exactly: Vanguard local authentication verified")
    .content
)

## Cloud-native identity protocol (`DefaultAzureCredential`)

In [ ]:
print(
    build_chat_model("identity")
    .invoke("Reply with exactly: Vanguard identity authentication verified")
    .content
)